# Best model

## 1. Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import warnings, sys

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')
from colorama import Fore, Style


In [ ]:
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

In [37]:
from hydrosense.params import *
from hydrosense.database.bigquery import load_pem_bq, info_piezo, load_plean
from hydrosense.preprocess.preprocessor import preprocess_week, preprocess_week_w_PU_synth, split_data, split_lagged_data, make_preproc_week
from hydrosense.preprocess.cleaning import clean_piezo

from hydrosense.utils.evap import estim_PU

from hydrosense.interface.main import evaluate, evaluate_deeper, preprocess, train
from hydrosense.ml_logic.folding import get_folds
from hydrosense.ml_logic import base


In [52]:
FEATURE_COLS = ["semaine_sin","semaine_cos", "PU_synth", "PC1", "PC2", "PC3"]
FEATURE_COLS = ["semaine_sin","semaine_cos", "PU_synth"]
DATA_CODE_PIEZO = "BSS001QHYH"
df = load_plean(DATA_CODE_PIEZO)

TRAIN_END  = "2025-02-28"
TEST_START = "2025-03-01"
TEST_END   = "2025-05-31"

✅ BSS001QHYH : 11821 lignes de features ML chargées.


In [53]:
df.head()

,bss_id,date_mesure,niveau_nappe_eau,RR_synth,TM_synth,FFM_synth,PU_synth,PC1,PC2,PC3
0,BSS001QHYH,1994-01-18,14.20,0.0,-1.6,2.3,-0.761,5.529467,-0.907712,0.465264
1,BSS001QHYH,1994-01-19,14.20,0.0,-2.4,1.3,-0.520,5.507861,-0.903470,0.462943
2,BSS001QHYH,1994-01-20,14.13,1.4,1.6,1.6,0.677,5.480702,-0.894308,0.457180
3,BSS001QHYH,1994-01-21,14.07,0.8,3.5,1.4,0.070,5.442173,-0.881357,0.449040
4,BSS001QHYH,1994-01-22,14.00,0.0,5.7,0.5,-0.425,5.387199,-0.863381,0.437800


In [54]:
df_week = preprocess_week(df)

X_train_df, X_test_df, y_train_df, y_test_df = split_data(df_week, FEATURE_COLS, TARGET_COL, TRAIN_END, TEST_START, TEST_END )
X_train_df, y_train_df


⭐️ Use case: preprocess
✅ Moyenne hebdomadaire — 1689 semaines | 10 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 1623 | Test : 13



(             PU_synth  semaine_sin  semaine_cos
 date_mesure                                    
 1994-01-23      0.484     0.354605     0.935016
 1994-01-30     -1.878     0.464723     0.885456
 1994-02-06     22.949     0.568065     0.822984
 1994-02-13     16.082     0.663123     0.748511
 1994-02-20      4.810     0.748511     0.663123
 ...               ...          ...          ...
 2025-01-26      3.733     0.464723     0.885456
 2025-02-02     21.363     0.568065     0.822984
 2025-02-09      5.558     0.663123     0.748511
 2025-02-16     -1.079     0.748511     0.663123
 2025-02-23     -0.364     0.822984     0.568065
 
 [1623 rows x 3 columns],
 date_mesure
 1994-01-23    14.090000
 1994-01-30    13.707143
 1994-02-06    13.662857
 1994-02-13    13.732857
 1994-02-20    13.657143
                 ...    
 2025-01-26    13.198571
 2025-02-02    13.342857
 2025-02-09    13.227143
 2025-02-16    13.088571
 2025-02-23    13.010000
 Freq: W-SUN, Name: niveau_nappe_eau, Length: 1

In [55]:
df_week

,niveau_nappe_eau,RR_synth,PU_synth,TM_synth,PC1,PC2,PC3,FFM_synth,semaine_sin,semaine_cos
date_mesure,,,,,,,,,,
1994-01-23,14.090000,5.20,0.484,2.400000,5.437909,-0.878704,0.447234,1.766667,0.354605,0.935016
1994-01-30,13.707143,11.20,-1.878,7.842857,4.741158,-0.606062,0.271497,4.785714,0.464723,0.885456
1994-02-06,13.662857,33.60,22.949,6.485714,3.962465,-0.229476,0.021721,3.571429,0.568065,0.822984
1994-02-13,13.732857,27.40,16.082,5.885714,4.112225,-0.265872,0.043006,3.814286,0.663123,0.748511
1994-02-20,13.657143,17.20,4.810,5.885714,4.193190,-0.282091,0.051946,4.200000,0.748511,0.663123
...,...,...,...,...,...,...,...,...,...,...
2026-05-03,12.151429,12.92,-13.622,16.757143,0.644676,1.162306,-0.404748,2.842857,0.822984,-0.568065
2026-05-10,12.118571,18.34,-8.068,15.805714,0.363734,1.193812,-0.237296,2.785714,0.748511,-0.663123
2026-05-17,12.105714,10.86,-14.000,12.442857,0.149368,1.245580,-0.081569,3.514286,0.663123,-0.748511


In [56]:
# TRAIN / VAL Folding
# TODO for weekly folding
splits_ml = get_folds(X_train_df.index, n_splits=25, min_train_years=3 , val_months_duration= 3)

len(splits_ml)
len(splits_ml[0])
splits_ml[0]

print('Validation index span : ')
X_train_df.index[splits_ml[10][1]]

Validation index span : 


DatetimeIndex(['2011-03-06', '2011-03-13', '2011-03-20', '2011-03-27',
               '2011-04-03', '2011-04-10', '2011-04-17', '2011-04-24',
               '2011-05-01', '2011-05-08', '2011-05-15', '2011-05-22',
               '2011-05-29'],
              dtype='datetime64[us]', name='date_mesure', freq=None)

In [57]:
def cal_rmsse(y_true, y_pred, y_train_hist):
    # Erreur quadratique moyenne modèle
    numerator = np.mean((y_true - y_pred) ** 2)

    # Erreur quadratique moyenne de la persistance sur le train (Le bas)
    # np.diff(y_train_hist) fait la soustraction entre la semaine T et la semaine T-1
    denominator = np.mean(np.diff(y_train_hist) ** 2)
    return np.sqrt(numerator / denominator)

In [62]:
all_results = []  # Stocke les résultats pour chaque piezo

# temporary loop - - - -
for bss in TARGETS_BSS[-1:]:
    # TODO pipeline make_preproc_week
    # en attendant voici un petit loop manuel.

    df = load_plean(bss)
    df_week = preprocess_week(df)
    X_train_df, X_test_df, y_train_df, y_test_df = split_data(df_week,
                                                              FEATURE_COLS, TARGET_COL,
                                                              TRAIN_END, TEST_START, TEST_END
                                                              )

    base1, _  = train(X_train_df, y_train_df,
                    pick_model = 'BASE',
                    optimize = False
                    )



    Xt, Xe, yt, ye, scaler = make_preproc_week(df, FEATURE_COLS, TARGET_COL, TRAIN_END, TEST_START, TEST_END)


    base, _ = train(Xt, yt, pick_model = 'BASE', optimize = False
                    )

    lasso, _ = train(Xt, yt, pick_model= 'LASSO' , optimize= False  )


✅ BSS002EDYK : 12261 lignes de features ML chargées.

⭐️ Use case: preprocess
✅ Moyenne hebdomadaire — 1753 semaines | 10 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 1688 | Test : 13

⏳ Initialisation de la persistance annuelle (J-365)...
✅ train() done — Baseline prête.

⭐️ Use case: preprocess
✅ Moyenne hebdomadaire — 1753 semaines | 10 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 1688 | Test : 13

Adimensionnement des features (Standardisation)...
✅ Adimensionnement terminé : 9 en Standard, 0 en MinMax.
⏳ Initialisation de la persistance annuelle (J-365)...
✅ train() done — Baseline prête.

Use case: train (Mode Baseline Lasso)
✅ train() done — Modèle Lasso entraîné.



After splits and preproc -- a litte baseline sounds good

In [60]:
display(X_train_df)
display(Xt)

,PU_synth,semaine_sin,semaine_cos
date_mesure,,,
1992-10-25,20.652,-0.885456,0.464723
1992-11-01,55.003,-0.822984,0.568065
1992-11-08,0.481,-0.748511,0.663123
1992-11-15,32.608,-0.663123,0.748511
1992-11-22,32.135,-0.568065,0.822984
...,...,...,...
2025-01-26,4.244,0.464723,0.885456
2025-02-02,31.866,0.568065,0.822984
2025-02-09,10.176,0.663123,0.748511


,semaine_sin,semaine_cos,niveau_nappe_eau_lag_1,niveau_nappe_eau_lag_2,niveau_nappe_eau_lag_3,niveau_nappe_eau_lag_4,niveau_nappe_eau_lag_52,PU_synth_lag_1,PU_synth_lag_2,PU_synth_lag_3,PU_synth_lag_4
date_mesure,,,,,,,,,,,
1993-10-24,-0.935016,0.354605,0.310667,0.340967,-0.493818,-1.123316,2.199645,0.188537,-0.184297,2.251912,3.051786
1993-10-31,-0.885456,0.464723,-0.075980,0.311042,0.341980,-0.492455,2.630690,0.011122,0.188488,-0.185163,2.245463
1993-11-07,-0.822984,0.568065,-0.154230,-0.075678,0.312048,0.343321,2.111615,-0.288854,0.011070,0.187061,-0.185826
1993-11-14,-0.748511,0.663123,-0.069076,-0.153943,-0.074768,0.313389,1.235108,0.709338,-0.288909,0.009911,0.185514
1993-11-21,-0.663123,0.748511,-0.009238,-0.068772,-0.153052,-0.073416,2.056976,-0.133324,0.709295,-0.289616,0.008784
...,...,...,...,...,...,...,...,...,...,...,...
2025-01-26,0.464723,0.885456,0.959681,0.771424,0.650512,0.831432,0.825312,-0.164728,1.179342,-0.049794,-0.209107
2025-02-02,0.568065,0.822984,0.747946,0.960180,0.772543,0.651844,0.838972,0.145488,-0.164781,1.176422,-0.050779
2025-02-09,0.663123,0.748511,1.081659,0.748405,0.961346,0.773872,0.679607,1.254715,0.145439,-0.165675,1.172526


In [ ]:
# A single PIEZO - TEST - on the basem
# - - - - - -
y_pred_baseline = base.predict(X_test_df)
print('erreur baseline.')
error1 = cal_rmsse(y_test_df.values, y_pred_baseline, y_train_df.values)

# Evaluate function from main should work here
error2 = cal_rmsse(ye.values, y_pred_baseline, yt.values)
print(error1, error2)

erreur baseline.
1.8170399995286877 1.8289095117360281


In [67]:
y_pred = lasso.predict(Xe)
y_pred, y_pred_baseline

(array([27.20288512, 27.15010849, 27.09417325, 27.15670123, 27.02340142,
        26.97013985, 26.90896277, 26.86093099, 27.22524929, 27.0022978 ,
        27.00964924, 27.09479202, 26.98257347]),
 array([27.69714286, 27.76428571, 27.73285714, 27.51142857, 27.49142857,
        27.48      , 27.35142857, 27.24571429, 27.15      , 27.09857143,
        27.08      , 27.06142857, 27.09142857]))

In [68]:
for bss in TARGETS_BSS:

    print(f"\n{'═'*55}")
    print(f"🔄 Traitement : {bss}")
    print(f"{'═'*55}")

    try:
        # ── Chargement & preprocessing ───────────────────────────
        df = load_plean(bss)
        df_week = preprocess_week_w_PU_synth(df)
        X_train_df, X_test_df, y_train_df, y_test_df = split_data(df_week)

        print(f"   Train : {len(X_train_df)} sem. | Test : {len(X_test_df)} sem.")

        # ── Baseline A : Persistance immédiate ───────────────────
        valeur_derniere = y_train_df.iloc[-1]
        y_pred_baseline_hebdo = np.full(len(y_test_df), fill_value=valeur_derniere)

        # ── Baseline B : Persistance saisonnière (J-52) ──────────
        y_pred_baseline_saisonniere = []
        for date in y_test_df.index:
            date_an_dernier = date - pd.Timedelta(weeks=52)
            idx = y_train_df.index.get_indexer([date_an_dernier], method='nearest')[0]
            y_pred_baseline_saisonniere.append(y_train_df.iloc[idx])
        y_pred_baseline_saisonniere = np.array(y_pred_baseline_saisonniere)

        # ── SARIMAX ───────────────────────────────────────────────
        try:
            model_sarimax = SARIMAX(
                y_train_df,
                order=(1, 0, 1),
                seasonal_order=(1, 0, 0, 52),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            preds_sarimax = model_sarimax.fit(disp=False).predict(
                start=len(y_train_df),
                end=len(y_train_df) + len(y_test_df) - 1
            ).values
        except Exception as e:
            print(f"   ⚠️ SARIMAX échoué ({e}) — remplacé par NaN")
            preds_sarimax = np.full(len(y_test_df), np.nan)

        # ── XGBoost best params ───────────────────────────────────
        best_xgb = XGBRegressor(
            colsample_bytree=0.6, learning_rate=0.1, max_depth=2,
            min_child_weight=5, n_estimators=500, subsample=0.8,
            random_state=42, n_jobs=-1, verbosity=0
        )
        best_xgb.fit(X_train_df, y_train_df)

        # ── Dictionnaire des modèles ──────────────────────────────
        models = {
            'Baseline (Semaine Dernière)': y_pred_baseline_hebdo,
            'Baseline (Année Dernière)':   y_pred_baseline_saisonniere,
            'SARIMAX(1,0,1)(1,0,0)[52]':   preds_sarimax,
            'Lasso':                        Lasso(alpha=0.01, random_state=42),
            'ElasticNet':                   ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42),
            'Random Forest':                RandomForestRegressor(n_estimators=300, max_depth=5,random_state=42, n_jobs=-1),
            'XGBoost Optimisé':             best_xgb,
        }

        # ── Calcul des métriques ──────────────────────────────────
        for name, m in models.items():
            if isinstance(m, np.ndarray):
                preds = m
            else:
                if name != 'XGBoost Optimisé':
                    m.fit(X_train_df, y_train_df)
                preds = m.predict(X_test_df)

            if np.any(np.isnan(preds)):
                mae, rmsse = np.nan, np.nan
            else:
                mae   = mean_absolute_error(y_test_df, preds)
                rmsse = cal_rmsse(y_test_df, preds, y_train_df)

            all_results.append({
                'BSS':         bss,
                'Modèle':      name,
                'MAE (m NGF)': round(mae, 4) if not np.isnan(mae) else np.nan,
                'RMSSE':       round(rmsse, 4) if not np.isnan(rmsse) else np.nan,
                'n_test':      len(y_test_df),
            })

        print(f"   ✅ OK")

    except Exception as e:
        print(f"   ❌ Erreur sur {bss} : {e}")
        continue

# ── Tableau global ────────────────────────────────────────────────────────────
df_all = pd.DataFrame(all_results)

# Vue synthétique : MAE moyenne par modèle sur tous les piézos
df_summary = (
    df_all.groupby('Modèle')[['MAE (m NGF)', 'RMSSE']]
    .mean()
    .round(4)
    .sort_values('MAE (m NGF)')
)

print("\n\n🏆 SYNTHÈSE TOUS PIÉZOS — MAE & RMSSE moyens")
print("═" * 55)
display(df_summary.style.background_gradient(cmap='RdYlGn_r'))

# Vue détaillée : tous les piézos × tous les modèles
df_pivot = df_all.pivot(index='BSS', columns='Modèle', values='MAE (m NGF)').round(4)
print("\n📊 MAE DÉTAILLÉE PAR PIÉZO ET MODÈLE")
display(df_pivot.style.background_gradient(cmap='RdYlGn_r', axis=1))


═══════════════════════════════════════════════════════
🔄 Traitement : BSS002DEZW
═══════════════════════════════════════════════════════
✅ BSS002DEZW : 10370 lignes de features ML chargées.

⭐️ Use case: preprocess_week_w_PU_synth
✅ preprocess() done — 0 semaines | 15 colonnes

   ❌ Erreur sur BSS002DEZW : split_data() missing 5 required positional arguments: 'feature_cols', 'target_col', 'train_end', 'test_start', and 'test_end'

═══════════════════════════════════════════════════════
🔄 Traitement : BSS000ZPHJ
═══════════════════════════════════════════════════════


KeyboardInterrupt: 

In [ ]:
df_week

,niveau_nappe_eau,semaine_sin,semaine_cos,lag_1,lag_2,lag_3,lag_4,lag_52,PC1_lag_1,PC2_lag_1,PC3_lag_1,PU_synth_lag_1,PU_synth_lag_2,PU_synth_lag_3,PU_synth_lag_4
date_mesure,,,,,,,,,,,,,,,
1993-10-24,26.784286,-0.935016,0.354605,27.024286,27.042857,26.524286,26.132857,28.206667,2.534364,0.300811,0.573558,0.759429,-0.566714,8.116000,10.995571
1993-10-31,26.735714,-0.885456,0.464723,26.784286,27.024286,27.042857,26.524286,28.477143,2.600164,0.308618,0.588449,0.128286,0.759429,-0.566714,8.116000
1993-11-07,26.788571,-0.822984,0.568065,26.735714,26.784286,27.024286,27.042857,28.151429,3.201027,0.379916,0.724432,-0.938857,0.128286,0.759429,-0.566714
1993-11-14,26.825714,-0.748511,0.663123,26.788571,26.735714,26.784286,27.024286,27.601429,3.583417,0.425289,0.810971,2.612143,-0.938857,0.128286,0.759429
1993-11-21,26.844286,-0.663123,0.748511,26.825714,26.788571,26.735714,26.784286,28.117143,3.919441,0.465161,0.887018,-0.385571,2.612143,-0.938857,0.128286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-26,27.045714,0.885456,-0.464723,27.118571,27.151429,27.201429,27.278571,27.065714,2.180096,1.912442,-0.826843,-2.545429,-1.234000,-1.185714,-2.054143
2026-05-03,27.011429,0.822984,-0.568065,27.045714,27.118571,27.151429,27.201429,27.104286,1.826535,1.884814,-0.736395,-3.108571,-2.545429,-1.234000,-1.185714
2026-05-10,27.047143,0.748511,-0.663123,27.011429,27.045714,27.118571,27.151429,27.108571,1.204911,1.720551,-0.637630,1.646857,-3.108571,-2.545429,-1.234000


In [ ]:
from hydrosense.api.fast import predict

ModuleNotFoundError: No module named 'fastapi'

In [ ]:
from hydrosense.interface.main import pred_future
from hydrosense.preprocess.preprocessor import make_preproc_week

In [ ]:
df_clean = df

TypeError: make_preproc_week() missing 5 required positional arguments: 'feature_cols', 'target_col', 'train_end', 'test_start', and 'test_end'

In [ ]:
Xt

,semaine_sin,semaine_cos,niveau_nappe_eau_lag_1,niveau_nappe_eau_lag_2,niveau_nappe_eau_lag_3,niveau_nappe_eau_lag_4,niveau_nappe_eau_lag_52,PC1_lag_1,PC2_lag_1,PC3_lag_1,PU_synth_lag_1,PU_synth_lag_2,PU_synth_lag_3,PU_synth_lag_4
date_mesure,,,,,,,,,,,,,,
1995-01-22,0.354605,0.935016,0.886647,1.211597,1.122904,1.086431,1.574335,1.248035,-0.545371,-1.017346,-0.209303,0.155994,1.997813,0.455964
1995-01-29,0.464723,0.885456,1.046424,0.886087,1.211224,1.122724,1.254739,1.043644,-0.094470,-1.078593,5.136680,-0.209297,0.154840,1.999129
1995-02-05,0.568065,0.822984,1.784786,1.045817,0.885769,1.211037,1.217770,1.809043,-1.668730,-1.132966,1.033369,5.136690,-0.210014,0.155395
1995-02-12,0.663123,0.748511,1.502756,1.783963,1.045472,0.885609,1.276204,1.931136,-1.912351,-0.727999,0.179127,1.033376,5.129565,-0.209609
1995-02-19,0.748511,0.663123,1.263091,1.502015,1.783494,1.045298,1.213000,1.687206,-1.297973,-0.433576,0.991738,0.179133,1.031170,5.132176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-26,0.464723,0.885456,1.289720,1.304773,0.605078,0.511790,0.918447,1.662348,-1.772644,-0.219139,-0.113997,2.068273,1.239975,-0.026728
2025-02-02,0.568065,0.822984,0.833388,1.289042,1.304385,0.604943,0.709756,1.302444,-0.743648,-0.431813,0.392972,-0.113991,2.064826,1.240979
2025-02-09,0.663123,0.748511,0.955642,0.832843,1.288656,1.304189,0.656092,1.206784,-0.426563,-0.416212,1.228912,0.392978,-0.114822,2.066170


In [ ]:



Xtot = pd.concat([Xt, Xe])
ytot = pd.concat([yt, ye])


model, _ = train(Xtot, ytot, optimize=False)

forecast = pred_future(model, Xtot, n_weeks=13)

print(forecast)

✅ BSS001QHYH : 11821 lignes de features ML chargées.

⭐️ Use case: preprocess
✅ preprocess() done — 1689 semaines | 10 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 1623 | Test : 13

Adimensionnement des features (Standardisation)...
✅ Adimensionnement terminé : 12 en Standard, 0 en MinMax.

Use case: train (Mode Baseline Lasso)
✅ train() done — Modèle Lasso entraîné.


⭐️ Use case: pred_future
✅ Predicted 1 values


KeyError: 'niveau_nappe_eau'

In [ ]:
TRAINING_FEATURES = [
    'semaine_sin', 'semaine_cos',
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_52',
    'PC1_lag_1', 'PC2_lag_1', 'PC3_lag_1',
    'PU_synth_lag_1', 'PU_synth_lag_2', 'PU_synth_lag_3', 'PU_synth_lag_4'
]


def predict_recursive(model, df_week: pd.DataFrame, n_weeks: int) -> np.ndarray:

    pu_seasonal = (
        df_week.groupby(df_week.index.isocalendar().week.astype(int))['PU_synth_lag_1']
        .mean().to_dict()
    )

    df_future = df_week.loc[:params.TEST_START_DATE].copy()
    predictions = []

    for i in range(n_weeks):
        last_row  = df_future[TRAINING_FEATURES].tail(1).values
        y_next    = float(model.predict(last_row)[0])
        next_date = df_future.index[-1] + pd.Timedelta(weeks=1)

        next_semaine = int(next_date.isocalendar().week)
        pu_future    = pu_seasonal.get(next_semaine, df_week['PU_synth_lag_1'].mean())

        new_row = pd.DataFrame({
            "niveau_nappe_eau": [y_next],
            "semaine_sin":      [np.sin(2 * np.pi * next_semaine / 52)],
            "semaine_cos":      [np.cos(2 * np.pi * next_semaine / 52)],
            # lags niveau — décalage en cascade
            "lag_1":            [y_next],
            "lag_2":            [df_future["lag_1"].iloc[-1]],
            "lag_3":            [df_future["lag_2"].iloc[-1]],
            "lag_4":            [df_future["lag_3"].iloc[-1]],
            "lag_52":           [df_future["niveau_nappe_eau"].iloc[-51]],
            # PC : on reporte les dernières valeurs connues
            "PC1_lag_1":        [df_future["PC1_lag_1"].iloc[-1]],
            "PC2_lag_1":        [df_future["PC2_lag_1"].iloc[-1]],
            "PC3_lag_1":        [df_future["PC3_lag_1"].iloc[-1]],
            # PU_synth : proxy saisonnier + décalage en cascade
            "PU_synth_lag_1":   [pu_future],
            "PU_synth_lag_2":   [df_future["PU_synth_lag_1"].iloc[-1]],
            "PU_synth_lag_3":   [df_future["PU_synth_lag_2"].iloc[-1]],
            "PU_synth_lag_4":   [df_future["PU_synth_lag_3"].iloc[-1]],
        }, index=[next_date])

        df_future = pd.concat([df_future, new_row])
        predictions.append(y_next)

    return np.array(predictions)

In [ ]:
all_results = []

for bss in params.TARGETS_BSS:
    print(f"\n{'═'*55}")
    print(f"🔄 Traitement : {bss}")
    print(f"{'═'*55}")

    try:
        # ── Chargement & preprocessing ───────────────────────────
        df      = load_plean(bss)
        df_week = preprocess_week_w_PU_synth(df)
        X_train_df, X_test_df, y_train_df, y_test_df = split_data(df_week)

        print(f"   Train : {len(X_train_df)} sem. | Test : {len(X_test_df)} sem.")

        n_test = len(y_test_df)

        # ── Baseline A : Persistance immédiate ───────────────────
        y_pred_baseline_hebdo = np.full(n_test, fill_value=y_train_df.iloc[-1])

        # ── Baseline B : Persistance saisonnière (J-52) ──────────
        y_pred_baseline_saisonniere = []
        for date in y_test_df.index:
            date_an_dernier = date - pd.Timedelta(weeks=52)
            idx = y_train_df.index.get_indexer([date_an_dernier], method='nearest')[0]
            y_pred_baseline_saisonniere.append(y_train_df.iloc[idx])
        y_pred_baseline_saisonniere = np.array(y_pred_baseline_saisonniere)

        # ── SARIMAX — autorégressif par nature ───────────────────
        try:
            model_sarimax = SARIMAX(
                y_train_df,
                order=(1, 0, 1),
                seasonal_order=(1, 0, 0, 52),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            preds_sarimax = model_sarimax.fit(disp=False).predict(
                start=len(y_train_df),
                end=len(y_train_df) + n_test - 1
            ).values
        except Exception as e:
            print(f"   ⚠️ SARIMAX échoué ({e}) — remplacé par NaN")
            preds_sarimax = np.full(n_test, np.nan)

        # ── XGBoost — fit sur train uniquement ───────────────────
        best_xgb = XGBRegressor(
            colsample_bytree=0.6, learning_rate=0.1, max_depth=2,
            min_child_weight=5, n_estimators=500, subsample=0.8,
            random_state=42, n_jobs=-1, verbosity=0
        )
        best_xgb.fit(X_train_df, y_train_df)

        # ── Dictionnaire des modèles ──────────────────────────────
        models = {
            'Baseline (Semaine Dernière)': y_pred_baseline_hebdo,          # np.ndarray
            'Baseline (Année Dernière)':   y_pred_baseline_saisonniere,    # np.ndarray
            'SARIMAX(1,0,1)(1,0,0)[52]':   preds_sarimax,                  # np.ndarray
            'Lasso':                        Lasso(alpha=0.01, random_state=42),
            'ElasticNet':                   ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42),
            'Random Forest':                RandomForestRegressor(n_estimators=300, max_depth=5,
                                                random_state=42, n_jobs=-1),
            'XGBoost Optimisé':             best_xgb,
        }

        # ── Calcul des métriques ──────────────────────────────────
        for name, m in models.items():

            if isinstance(m, np.ndarray):
                # Baselines & SARIMAX — prédictions déjà calculées
                preds = m
            else:
                # Modèles ML — fit si pas encore entraîné
                if name != 'XGBoost Optimisé':
                    m.fit(X_train_df, y_train_df)

                # ← Prédiction récursive : un seul point à la fois,
                #   jamais les features du test
                preds = predict_recursive(m, df_week, n_weeks=n_test)

            if np.any(np.isnan(preds)):
                mae, rmsse_val = np.nan, np.nan
            else:
                mae       = mean_absolute_error(y_test_df, preds)
                rmsse_val = cal_rmsse(y_test_df, preds, y_train_df)

            all_results.append({
                'BSS':         bss,
                'Modèle':      name,
                'MAE (m NGF)': round(mae, 4)       if not np.isnan(mae)       else np.nan,
                'RMSSE':       round(rmsse_val, 4) if np.isfinite(rmsse_val)  else np.nan,
                'n_test':      n_test,
            })

        print(f"   ✅ OK")

    except Exception as e:
        print(f"   ❌ Erreur sur {bss} : {e}")
        continue

# ── Tableau global ─────────────────────────────────────────────────────────────
df_all = pd.DataFrame(all_results)

df_summary = (
    df_all.groupby('Modèle')[['MAE (m NGF)', 'RMSSE']]
    .mean().round(4).sort_values('MAE (m NGF)')
)
print("\n\n🏆 SYNTHÈSE TOUS PIÉZOS — MAE & RMSSE moyens")
print("═" * 55)
display(df_summary.style.background_gradient(cmap='RdYlGn_r'))

df_pivot = df_all.pivot(index='BSS', columns='Modèle', values='MAE (m NGF)').round(4)
print("\n📊 MAE DÉTAILLÉE PAR PIÉZO ET MODÈLE")
display(df_pivot.style.background_gradient(cmap='RdYlGn_r', axis=1))


═══════════════════════════════════════════════════════
🔄 Traitement : BSS002DEZW
═══════════════════════════════════════════════════════
✅ BSS002DEZW : 10370 lignes de features ML chargées.

⭐️ Use case: preprocess_week_w_PU_synth
✅ preprocess() done — 0 semaines | 15 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 0 | Test : 0

   Train : 0 sem. | Test : 0 sem.
   ❌ Erreur sur BSS002DEZW : single positional indexer is out-of-bounds

═══════════════════════════════════════════════════════
🔄 Traitement : BSS000ZPHJ
═══════════════════════════════════════════════════════
✅ BSS000ZPHJ : 7464 lignes de features ML chargées.

⭐️ Use case: preprocess_week_w_PU_synth
✅ preprocess() done — 933 semaines | 15 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 867 | Test : 13

   Train : 867 sem. | Test : 13 sem.
   ❌ Erreur sur BSS000ZPHJ : X has 14 features, but Lasso is expecting 2 features as input.

═══════════════════════════════════════════════════════
🔄 Tr

,MAE (m NGF),RMSSE
Modèle,,
"SARIMAX(1,0,1)(1,0,0)[52]",0.820200,2.569500
Baseline (Semaine Dernière),0.856400,2.758600
Baseline (Année Dernière),1.510000,4.695000



📊 MAE DÉTAILLÉE PAR PIÉZO ET MODÈLE


Modèle,Baseline (Année Dernière),Baseline (Semaine Dernière),"SARIMAX(1,0,1)(1,0,0)[52]"
BSS,,,
BSS000ZPHJ,0.471200,0.756900,0.657200
BSS000ZQXN,0.863700,1.358900,1.231000
BSS001PGUQ,1.009000,0.707600,0.633600
BSS001QHPU,1.100800,0.939300,0.598200
BSS001QHYH,0.509900,0.391300,0.248900
BSS001QSMT,1.531600,0.710200,0.804800
BSS001QTKG,5.216300,3.803400,3.821800
BSS001RQQE,1.115200,0.684000,0.582500
BSS001SHNE,2.062600,1.506300,1.291200


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error

# On s'assure d'avoir notre fonction de prédiction récursive
def pred_future_for_benchmark(model, X_train, y_train, X_test, n_weeks: int) -> np.array:
    """
    Simule une prédiction pas à pas (autorégressive) sur l'horizon de test
    pour les modèles de ML qui ne gèrent pas le futur nativ_state.
    """
    # Recréation d'un dataframe temporaire aligné avec tes features
    df_temp = X_train.copy()
    df_temp['niveau_nappe_eau'] = y_train

    # Liste ordonnée de tes colonnes de features
    features_list = list(X_train.columns)
    predictions = []

    for i in range(n_weeks):
        # 1. Prendre les features de la ligne à prédire
        # Pour les variables exogènes (météo, saisonnalité), on prend celles du bloc de test réel
        current_features = X_test.iloc[i].copy()

        # 2. Mettre à jour les lags dynamiques de la nappe issus de nos propres prédictions précédentes
        if i >= 1:
            current_features["lag_1"] = predictions[-1]
        if i >= 2:
            current_features["lag_2"] = predictions[-2]
        if i >= 3:
            current_features["lag_3"] = predictions[-3]
        if i >= 4:
            current_features["lag_4"] = predictions[-4]

        # Pour le lag 52 annuel, si on a déjà avancé de + de 52 semaines, on prend notre prédo, sinon l'historique train
        if i >= 52:
            current_features["lag_52"] = predictions[-52]

        # Convertir en matrice 2D pour scikit-learn / XGBoost
        row_vector = current_features[features_list].values.reshape(1, -1)

        # Prédire le point suivant
        y_next = float(model.predict(row_vector)[0])
        predictions.append(y_next)

    return np.array(predictions)


all_results = []

for bss in params.TARGETS_BSS:
    print(f"\n{'═'*55}")
    print(f"🔄 Traitement : {bss}")
    print(f"{'═'*55}")

    try:
        # ── 1. Chargement & preprocessing ───────────────────────────
        df = load_plean(bss)
        df_week = preprocess_week_w_PU_synth(df)
        X_train_df, X_test_df, y_train_df, y_test_df = split_data(df_week)

        print(f"   Train : {len(X_train_df)} sem. | Test : {len(X_test_df)} sem.")
        n_test = len(y_test_df)

        if n_test == 0:
            print(f"   ⚠️ Pas assez de données de test pour {bss}.")
            continue

        # ── 2. Baselines de référence ───────────────────────────────
        # Baseline A : Persistance immédiate
        y_pred_baseline_hebdo = np.full(n_test, fill_value=y_train_df.iloc[-1])

        # Baseline B : Persistance saisonnière (J-52)
        y_pred_baseline_saisonniere = []
        for date in y_test_df.index:
            date_an_dernier = date - pd.Timedelta(weeks=52)
            idx = y_train_df.index.get_indexer([date_an_dernier], method='nearest')[0]
            y_pred_baseline_saisonniere.append(y_train_df.iloc[idx])
        y_pred_baseline_saisonniere = np.array(y_pred_baseline_saisonniere)

        # Dictionnaire temporaire pour stocker les prédictions de ce piézomètre
        dict_preds = {
            'Semaine Dernière': y_pred_baseline_hebdo,
            'Persistance Saisonnière': y_pred_baseline_saisonniere
        }

        # ── 3. Modèle Temporel : SARIMAX ─────────────────────────────
        try:
            model_sarimax = SARIMAX(
                y_train_df,
                order=(1, 0, 1),
                seasonal_order=(1, 0, 0, 52),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            dict_preds['SARIMAX'] = model_sarimax.fit(disp=False).predict(
                start=len(y_train_df),
                end=len(y_train_df) + n_test - 1
            ).values
        except Exception as e:
            print(f"   ⚠️ Échec SARIMAX sur {bss}: {e}")
            dict_preds['SARIMAX'] = np.full(n_test, np.nan)

        # ── 4. Modèles de Machine Learning ───────────────────────────
        # Définition des instances de modèles Scikit-Learn & XGBoost natifs
        models_to_test = {
            'Lasso': Lasso(alpha=0.01, random_state=42),
            'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42),
            'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42),
            'XGBoost': XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
        }

        for name, model_obj in models_to_test.items():
            try:
                # Entraînement sur l'historique
                model_obj.fit(X_train_df, y_train_df)

                # Prédiction itérative pas à pas dans le futur (éviter le data leakage)
                dict_preds[name] = pred_future_for_benchmark(
                    model_obj, X_train_df, y_train_df, X_test_df, n_weeks=n_test
                )
            except Exception as e:
                print(f"   ⚠️ Échec modèle {name} sur {bss}: {e}")
                dict_preds[name] = np.full(n_test, np.nan)

        # ── 5. Calcul des métriques & agrégation ─────────────────────
        for name, preds in dict_preds.items():
            if np.any(np.isnan(preds)) or len(preds) == 0:
                mae, rmsse_val = np.nan, np.nan
            else:
                mae = mean_absolute_error(y_test_df, preds)
                rmsse_val = cal_rmsse(y_test_df, preds, y_train_df)

            all_results.append({
                'BSS': bss,
                'Modèle': name,
                'MAE (m NGF)': round(mae, 4) if not np.isnan(mae) else np.nan,
                'RMSSE': round(rmsse_val, 4) if np.isfinite(rmsse_val) else np.nan,
                'n_test': n_test,
            })

        print(f"   ✅ OK")

    except Exception as e:
        print(f" ❌ Erreur critique sur {bss} : {e}")
        continue

# ── Tableau global final ──────────────────────────────────────────────────────
df_all = pd.DataFrame(all_results)

df_summary = (
    df_all.groupby('Modèle')[['MAE (m NGF)', 'RMSSE']]
    .mean().round(4).sort_values('MAE (m NGF)')
)
print("\n\n🏆 SYNTHÈSE TOUS PIÉZOS — MAE & RMSSE moyens")
print("═" * 55)
display(df_summary.style.background_gradient(cmap='RdYlGn_r'))

df_pivot = df_all.pivot(index='BSS', columns='Modèle', values='MAE (m NGF)').round(4)
print("\n📊 MAE DÉTAILLÉE PAR PIÉZO ET MODÈLE")
display(df_pivot.style.background_gradient(cmap='RdYlGn_r', axis=1))


═══════════════════════════════════════════════════════
🔄 Traitement : BSS002DEZW
═══════════════════════════════════════════════════════
✅ BSS002DEZW : 10370 lignes de features ML chargées.

⭐️ Use case: preprocess_week_w_PU_synth
✅ preprocess() done — 0 semaines | 15 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 0 | Test : 0

   Train : 0 sem. | Test : 0 sem.
   ⚠️ Pas assez de données de test pour BSS002DEZW.

═══════════════════════════════════════════════════════
🔄 Traitement : BSS000ZPHJ
═══════════════════════════════════════════════════════
✅ BSS000ZPHJ : 7464 lignes de features ML chargées.

⭐️ Use case: preprocess_week_w_PU_synth
✅ preprocess() done — 933 semaines | 15 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 867 | Test : 13

   Train : 867 sem. | Test : 13 sem.
   ✅ OK

═══════════════════════════════════════════════════════
🔄 Traitement : BSS000ZQXN
═══════════════════════════════════════════════════════
✅ BSS000ZQXN : 6534 lignes

,MAE (m NGF),RMSSE
Modèle,,
Random Forest,0.531000,2.073500
XGBoost,0.532900,2.074400
ElasticNet,0.599100,2.245200
Lasso,0.599500,2.248000
SARIMAX,0.820200,2.569500
Semaine Dernière,0.856400,2.758600
Persistance Saisonnière,1.510000,4.695000



📊 MAE DÉTAILLÉE PAR PIÉZO ET MODÈLE


Modèle,ElasticNet,Lasso,Persistance Saisonnière,Random Forest,SARIMAX,Semaine Dernière,XGBoost
BSS,,,,,,,
BSS000ZPHJ,0.333700,0.336900,0.471200,0.199600,0.657200,0.756900,0.198600
BSS000ZQXN,0.594700,0.598300,0.863700,0.365200,1.231000,1.358900,0.373300
BSS001PGUQ,0.384300,0.387400,1.009000,0.271600,0.633600,0.707600,0.275100
BSS001QHPU,0.406300,0.406500,1.100800,0.240600,0.598200,0.939300,0.240800
BSS001QHYH,0.296100,0.295100,0.509900,0.091300,0.248900,0.391300,0.092000
BSS001QSMT,1.758700,1.755700,1.531600,1.824600,0.804800,0.710200,1.816600
BSS001QTKG,0.982300,0.992900,5.216300,0.553300,3.821800,3.803400,0.553900
BSS001RQQE,0.329200,0.328200,1.115200,0.257100,0.582500,0.684000,0.267700
BSS001SHNE,0.485700,0.484900,2.062600,0.455100,1.291200,1.506300,0.452400
